# Week 5 & 6 — Bioinformatics Pipeline (Variants, Phasing, Comparison)

**By:Zeina Ebeid**

This notebook is self-contained: it downloads inputs, installs tools, executes the pipeline, and embeds results. Running `jupyter nbconvert --to notebook --execute week5.ipynb --stdout` should reproduce all outputs.

**Genes of interest:** `CYP2C8`, `CYP2C9`, `CYP2C19` (drug metabolism).

**Sequencing technologies:** Illumina (short-read) and PacBio HiFi (long-read) from sample NA12878.


## Parameters
Edit if needed (e.g., subsampling size for speed).

In [ ]:
%%bash
set -euo pipefail

# You can tweak these for speed/CI
ILLUMINA_RUN="ERR194147"          # Illumina Platinum Genomes run for NA12878 (paired-end)
PACBIO_EXP="SRX5780566"           # NA12878 PacBio HiFi experiment accession (will resolve runs)
SUBSAMPLE_FQ="200000"             # number of reads to sample (Illumina per mate; PacBio total); set "" to disable
THREADS="${THREADS:-4}"
WORKDIR="${PWD}"
OUTDIR="${WORKDIR}/wk5out"
GENE_LIST="CYP2C8,CYP2C9,CYP2C19"
GENOME_BUILD="hg38"
CHR="chr10"                       # all CYP2C genes of interest are on chr10 (10q23 region)
PAD_BP=2000                       # pad gene intervals by this many bp on both sides
IGV_SNAP="1"                      # set to 0 to skip automated IGV snapshots (may require xvfb on CI)
mkdir -p "$OUTDIR"/{ref,reads,align,vcf,phase,igv,logs}


## Install tools (CI safe)
Installs core CLI tools needed for the pipeline.

In [ ]:
%%bash
set -euo pipefail

# Prefer micromamba if available; otherwise use apt where possible
if command -v micromamba >/dev/null 2>&1; then
  eval "$(micromamba shell hook -s bash)"
  micromamba create -y -n wk5 -c bioconda -c conda-forge \
    samtools bcftools minimap2 freebayes seqtk jq parallel \
    hapcut2 bedtools pigz curl coreutils python=3.11 \
    pysam
  micromamba activate wk5
else
  sudo apt-get update -y
  sudo apt-get install -y samtools bcftools minimap2 freebayes seqtk jq parallel curl bedtools python3-pip xvfb
  pip3 install pysam
  # hapcut2 may not be in apt; build if missing
  if ! command -v HAPCUT2 >/dev/null 2>&1; then
    git clone --depth=1 https://github.com/vibansal/HapCUT2.git
    make -C HapCUT2
    sudo install -m 0755 HapCUT2/build/HAPCUT2 /usr/local/bin/HAPCUT2
    sudo install -m 0755 HapCUT2/build/extractHAIRS /usr/local/bin/extractHAIRS
    sudo install -m 0755 HapCUT2/utilities/hapcut2vcf.py /usr/local/bin/hapcut2vcf.py || true
  fi
fi

# Log versions
samtools --version | head -n1
bcftools --version | head -n1
minimap2 --version
freebayes --version | head -n1 || true
HAPCUT2 2>/dev/null | head -n1 || true
python3 -c "import pysam,sys; print('pysam', pysam.__version__)"


## Download reference genome (GRCh38, chr10 only)

In [ ]:
%%bash
set -euo pipefail

REF="$OUTDIR/ref/${CHR}.fa"
if [ ! -s "$REF" ]; then
  echo "[ref] downloading ${CHR}.fa.gz from UCSC..."
  URL="https://hgdownload.cse.ucsc.edu/goldenpath/hg38/chromosomes/${CHR}.fa.gz"
  curl -sSL "$URL" -o "$OUTDIR/ref/${CHR}.fa.gz"
  gunzip -c "$OUTDIR/ref/${CHR}.fa.gz" > "$REF"
fi
samtools faidx "$REF"


## Get gene coordinates (Ensembl REST) and create padded BED

In [ ]:
%%bash
set -euo pipefail

BED="$OUTDIR/ref/genes.bed"
: > "$BED"
IFS=',' read -r -a GENES <<< "$GENE_LIST"
for G in "${GENES[@]}"; do
  JSON="$(curl -s 'https://rest.ensembl.org/lookup/symbol/homo_sapiens/'"$G"'?content-type=application/json')" || true
  CHR_JSON=$(echo "$JSON" | jq -r '.seq_region_name' || echo "")
  START=$(echo "$JSON" | jq -r '.start' || echo "0")
  END=$(echo "$JSON" | jq -r '.end' || echo "0")
  if [ "$CHR_JSON" != "null" ] && [ -n "$START" ] && [ -n "$END" ]; then
    if [[ "$CHR_JSON" != "MT" && "$CHR_JSON" != "chr"* ]]; then CHROM="chr${CHR_JSON}"; else CHROM="$CHR_JSON"; fi
    S=$((START - PAD_BP)); if [ $S -lt 1 ]; then S=1; fi
    E=$((END + PAD_BP))
    echo -e "${CHROM}\t${S}\t${E}\t${G}" >> "$BED"
    echo "[genes] $G -> ${CHROM}:${START}-${END} (+/- ${PAD_BP}bp)"
  fi
done
sort -k1,1 -k2,2n "$BED" -o "$BED"
cat "$BED"


## Download reads — Illumina (paired-end) and PacBio HiFi (NA12878)

In [ ]:
%%bash
set -euo pipefail

cd "$OUTDIR/reads"

# Illumina paired-end FASTQs from ENA (ERR194147)
ILLU_TSV="illumina.tsv"
curl -s "https://www.ebi.ac.uk/ena/portal/api/filereport?accession=${ILLUMINA_RUN}&result=read_run&fields=fastq_ftp&download=true" -o "$ILLU_TSV"
ILLU_FTPS=$(tail -n+2 "$ILLU_TSV" | tr ';' '\n' | sed 's/^/https:\/\//')
echo "$ILLU_FTPS" | nl

ILLU_R1=$(echo "$ILLU_FTPS" | sed -n '1p')
ILLU_R2=$(echo "$ILLU_FTPS" | sed -n '2p')
curl -L "$ILLU_R1" -o illumina_R1.fastq.gz
curl -L "$ILLU_R2" -o illumina_R2.fastq.gz

# Optional subsample for speed
if [ -n "${SUBSAMPLE_FQ}" ]; then
  seqtk sample -s100 <(pigz -dc illumina_R1.fastq.gz) ${SUBSAMPLE_FQ} | pigz > illumina_R1.sub.fastq.gz
  seqtk sample -s100 <(pigz -dc illumina_R2.fastq.gz) ${SUBSAMPLE_FQ} | pigz > illumina_R2.sub.fastq.gz
  ILL_R1="illumina_R1.sub.fastq.gz"; ILL_R2="illumina_R2.sub.fastq.gz"
else
  ILL_R1="illumina_R1.fastq.gz"; ILL_R2="illumina_R2.fastq.gz"
fi

# PacBio HiFi runs for experiment SRX5780566 (resolve to run accessions)
PB_TSV="pacbio.tsv"
curl -s "https://www.ebi.ac.uk/ena/portal/api/filereport?accession=${PACBIO_EXP}&result=read_run&fields=run_accession,fastq_ftp&download=true" -o "$PB_TSV"
# Take first non-empty fastq link to keep runtime reasonable
PB_FTP=$(awk -F'\t' 'NR>1 && $2!="" {split($2,a,";"); print a[1]; exit}' "$PB_TSV")
PB_URL="https://${PB_FTP}"
echo "[pacbio] Using: $PB_URL"
curl -L "$PB_URL" -o pacbio.fastq.gz

if [ -n "${SUBSAMPLE_FQ}" ]; then
  seqtk sample -s100 <(pigz -dc pacbio.fastq.gz) ${SUBSAMPLE_FQ} | pigz > pacbio.sub.fastq.gz
  PB_FQ="pacbio.sub.fastq.gz"
else
  PB_FQ="pacbio.fastq.gz"
fi

cd "$WORKDIR"


## Align reads with minimap2
- Illumina: `-ax sr`
- PacBio HiFi: `-ax map-hifi`

In [ ]:
%%bash
set -euo pipefail

REF="$OUTDIR/ref/${CHR}.fa"
cd "$OUTDIR/align"

# Illumina alignment
minimap2 -t "$THREADS" -ax sr "$REF" "$OUTDIR/reads/$ILL_R1" "$OUTDIR/reads/$ILL_R2" \
  | samtools sort -@ "$THREADS" -o illumina.sorted.bam
samtools index illumina.sorted.bam

# PacBio HiFi alignment
minimap2 -t "$THREADS" -ax map-hifi "$REF" "$OUTDIR/reads/$PB_FQ" \
  | samtools sort -@ "$THREADS" -o pacbio.sorted.bam
samtools index pacbio.sorted.bam

samtools flagstat illumina.sorted.bam | tee ../logs/illumina.flagstat.txt
samtools flagstat pacbio.sorted.bam | tee ../logs/pacbio.flagstat.txt

cd "$WORKDIR"


## Variant calling (bcftools) — restricted to gene intervals

In [ ]:
%%bash
set -euo pipefail

cd "$OUTDIR/vcf"
BED="$OUTDIR/ref/genes.bed"
REF="$OUTDIR/ref/${CHR}.fa"

# Illumina
bcftools mpileup -Ou -f "$REF" -R "$BED" "$OUTDIR/align/illumina.sorted.bam" \
  | bcftools call -mv -Ov -o illumina.raw.vcf
bcftools sort -Ov -o illumina.vcf illumina.raw.vcf
bgzip -f illumina.vcf
tabix -f -p vcf illumina.vcf.gz

# PacBio
bcftools mpileup -Ou -f "$REF" -R "$BED" "$OUTDIR/align/pacbio.sorted.bam" \
  | bcftools call -mv -Ov -o pacbio.raw.vcf
bcftools sort -Ov -o pacbio.vcf pacbio.raw.vcf
bgzip -f pacbio.vcf
tabix -f -p vcf pacbio.vcf.gz
cd "$WORKDIR"


## Phasing (HapCUT2) and conversion to phased VCF

In [ ]:
%%bash
set -euo pipefail

cd "$OUTDIR/phase"
REF="$OUTDIR/ref/${CHR}.fa"
BED="$OUTDIR/ref/genes.bed"

# Illumina
extractHAIRS --VCF "$OUTDIR/vcf/illumina.vcf.gz" --bam "$OUTDIR/align/illumina.sorted.bam" --out illumina.frag --regionList "$BED"
HAPCUT2 --fragments illumina.frag --VCF "$OUTDIR/vcf/illumina.vcf.gz" --output illumina.hapcut.txt --threads "$THREADS"
hapcut2vcf.py --vcf "$OUTDIR/vcf/illumina.vcf.gz" --hapcut illumina.hapcut.txt --output illumina.phased.vcf
bgzip -f illumina.phased.vcf && tabix -f -p vcf illumina.phased.vcf.gz

# PacBio
extractHAIRS --VCF "$OUTDIR/vcf/pacbio.vcf.gz" --bam "$OUTDIR/align/pacbio.sorted.bam" --out pacbio.frag --regionList "$BED"
HAPCUT2 --fragments pacbio.frag --VCF "$OUTDIR/vcf/pacbio.vcf.gz" --output pacbio.hapcut.txt --threads "$THREADS"
hapcut2vcf.py --vcf "$OUTDIR/vcf/pacbio.vcf.gz" --hapcut pacbio.hapcut.txt --output pacbio.phased.vcf
bgzip -f pacbio.phased.vcf && tabix -f -p vcf pacbio.phased.vcf.gz

cd "$WORKDIR"


## Compare phased VCFs and select discordant variants

In [ ]:
%%bash
set -euo pipefail

cd "$OUTDIR/vcf"
bcftools isec -p isec -n=2 illumina.phased.vcf.gz pacbio.phased.vcf.gz   # shared
bcftools isec -p isec_illumina_unique -n=1 -w1 illumina.phased.vcf.gz pacbio.phased.vcf.gz
bcftools isec -p isec_pacbio_unique  -n=1 -w2 illumina.phased.vcf.gz pacbio.phased.vcf.gz

echo "[compare] Shared variants: $(grep -vc '^#' isec/0000.vcf || true)"
echo "[compare] Illumina-unique:  $(grep -vc '^#' isec_illumina_unique/0000.vcf || true)"
echo "[compare] PacBio-unique:    $(grep -vc '^#' isec_pacbio_unique/0000.vcf || true)"

# Select up to 3 discordant variants (prefer Illumina-unique; fill from PacBio-unique if needed)
awk '!/^#/ {print $1":"$2}' isec_illumina_unique/0000.vcf 2>/dev/null | head -n 3 > ../phase/discordant.sites || true
awk '!/^#/ {print $1":"$2}' isec_pacbio_unique/0000.vcf  2>/dev/null | head -n 3 >> ../phase/discordant.sites || true
sort -u ../phase/discordant.sites -o ../phase/discordant.sites || true
cat ../phase/discordant.sites || true
cd "$WORKDIR"


## IGV screenshots (optional but recommended)
This section attempts automated IGV snapshots using headless X (`xvfb-run`) and IGV batch scripting. If it fails on CI, set `IGV_SNAP=0` above and do manual snapshots locally.

In [ ]:
%%bash
set -euo pipefail

if [ "${IGV_SNAP}" = "1" ]; then
  cd "$OUTDIR/igv"
  # Prepare IGV batch script
  cat > batch.igv <<'EOF'
new
genome ${GENOME}
snapshotDirectory ${SNAPDIR}
load ${ILLBAM}
load ${PBBAM}
# Zoom and snapshot each site
EOF
  GENOME="$OUTDIR/ref/${CHR}.fa"
  ILLBAM="$OUTDIR/align/illumina.sorted.bam"
  PBBAM="$OUTDIR/align/pacbio.sorted.bam"
  SNAPDIR="$OUTDIR/igv/snaps"
  mkdir -p "$SNAPDIR"
  while read -r LOC; do
    if [ -n "$LOC" ]; then
      printf "goto %s\n" "$LOC" >> batch.igv
      printf "snapshot %s.png\n" "$(echo "$LOC" | tr ':' '_')" >> batch.igv
    fi
  done < "$OUTDIR/phase/discordant.sites"
  
  # Download IGV if not present (command-line runnable .jar)
  if [ ! -s igv.jar ]; then
    curl -L https://data.broadinstitute.org/igv/projects/downloads/2.16/IGV_2.16.2.zip -o igv.zip || true
    unzip -o igv.zip || true
    find . -name "igv.jar" -print -quit | xargs -I{} cp {} ./igv.jar || true
  fi
  # Run IGV in batch mode (headless)
  if [ -s igv.jar ] && [ -s batch.igv ]; then
    xvfb-run -a java -Xmx4g -jar igv.jar -b batch.igv || true
  fi
  ls -l "$SNAPDIR" || true
  cd "$WORKDIR"
else
  echo "Skipping IGV automation as IGV_SNAP=0"
fi


## Python analysis: shared/unique counts and per-gene summaries

In [ ]:

import os, subprocess, json, pathlib, sys
from collections import defaultdict

out = os.environ["OUTDIR"]
paths = {
  "illumina_phased":"%s/phase/illumina.phased.vcf.gz" % out,
  "pacbio_phased":"%s/phase/pacbio.phased.vcf.gz" % out,
  "bed":"%s/ref/genes.bed" % out,
  "snaps":"%s/igv/snaps" % out,
}
print(paths)

def count_variants(vcf_gz):
    try:
        txt = subprocess.check_output(["bash","-lc", f"zgrep -vc '^#' {vcf_gz} || true"], text=True).strip()
        return int(txt) if txt else 0
    except Exception as e:
        return 0

counts = {k: count_variants(v) for k,v in paths.items() if k.endswith("phased")}
counts


### Helper: list phased variants per gene (for star-allele reasoning)

In [ ]:

# Produces a compact table (chrom, pos, ref, alt, GT phase) per gene per technology.
import subprocess, pandas as pd, io

bed = paths["bed"]
df_list = []
for tech, vcf in [("Illumina","%s/vcf/illumina.vcf.gz"%out), ("PacBio","%s/vcf/pacbio.vcf.gz"%out)]:
    cmd = f"bcftools view -R {bed} -H {vcf} | cut -f1-5"
    try:
        tsv = subprocess.check_output(["bash","-lc", cmd], text=True)
        sdf = pd.read_csv(io.StringIO(tsv), sep="\t", header=None, names=["CHROM","POS","ID","REF","ALT"])
        sdf["TECH"] = tech
        df_list.append(sdf)
    except:
        pass

if df_list:
    allv = pd.concat(df_list, ignore_index=True)
    # annotate gene label by interval overlap
    import pybedtools as pbt
    # If pybedtools isn't installed, skip gracefully
    try:
        from pybedtools import BedTool  # type: ignore
        vbed = BedTool.from_dataframe(allv[["CHROM","POS","POS","TECH"]].assign(POS2=lambda d:d["POS"])[["CHROM","POS","POS2","TECH"]])
        gbed = BedTool(bed)
        inter = vbed.intersect(gbed, wa=True, wb=True)
        # build mapping (this step is illustrative; exact join may vary)
    except Exception as e:
        pass

    # Save TSV for manual star-allele mapping
    allv.to_csv(f"{out}/phase/variants_for_star_alleles.tsv", sep="\t", index=False)
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Variants for star-allele interpretation", allv)
else:
    print("No variants parsed for helper table.")


## IGV screenshots & discussion
Include 2–3 discordant sites below (screenshots are saved to `wk5out/igv/snaps/`). Explain whether each is an artifact or true variant and why (e.g., strand bias, low support, context like homopolymers, mapping quality).

## Star-allele calls (PharmVar)
Using phased data, map observed variants in each gene to PharmVar star-allele definitions (e.g., `CYP2C19*2`, `*17`). Briefly justify (which SNVs/InDels on which haplotype) and state final diplotype per gene.

> Tip: Use the generated `variants_for_star_alleles.tsv` as a checklist. Cite PharmVar pages for each gene.

## Auto-check: did outputs get created?

In [ ]:
%%bash
set -euo pipefail

ok_ref=$(test -s "$OUTDIR/ref/${CHR}.fa" && echo ok || echo no)
ok_bam=$(test -s "$OUTDIR/align/illumina.sorted.bam" -a -s "$OUTDIR/align/pacbio.sorted.bam" && echo ok || echo no)
ok_vcf=$(test -s "$OUTDIR/vcf/illumina.vcf.gz" -a -s "$OUTDIR/vcf/pacbio.vcf.gz" && echo ok || echo no)
ok_phase=$(test -s "$OUTDIR/phase/illumina.phased.vcf.gz" -a -s "$OUTDIR/phase/pacbio.phased.vcf.gz" && echo ok || echo no)
ok_isec=$(test -s "$OUTDIR/vcf/isec/0000.vcf" && echo ok || echo no)

printf "**Reference:** %s\n**Alignment:** %s\n**Variant calling:** %s\n**Phasing:** %s\n**Comparison:** %s\n" \
  "$ok_ref" "$ok_bam" "$ok_vcf" "$ok_phase" "$ok_isec"


## Time estimate & AI use
- **Estimated time spent:** _fill in after completion_
- **AI tools used:** _list any prompts / tools used here (also include an `ai.md` in repo)_

## Appendix: CI setup
Use the provided GitHub Actions workflow (class repo) to execute this notebook in CI:
```
jupyter nbconvert --to notebook --execute week5/week5.ipynb --stdout
```
